In [1]:
import pandas as pd
import xgboost as xgb
from xgboost import XGBClassifier
import numpy as np 
from sklearn.model_selection import train_test_split, KFold, LeaveOneGroupOut, cross_val_score
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import balanced_accuracy_score
from cosmic import dna_props

In [14]:


def drop_bias(data: pd.DataFrame):
    """ 'WTtrinuc', 'mutTrinuc' are also objects, removes features causing bias"""
    bias_cols = ['merged', 'atom_site_occupancy']
    object_cols = data.select_dtypes(include=['object']).columns

    if not object_cols.empty:
        data = data.drop(columns=object_cols[-2:])
    
    # data.drop(columns=dna_props, inplace=True)
        

    data = data.drop(columns=bias_cols)
    return data
    
def select_equal_amounts(data: pd.DataFrame):
    """
    Randomly select an equal number of simulated variants for each chromosome
    based on the counts of human-derived variants.
    
    Args:
        data (pd.DataFrame): Input DataFrame with columns 'driver_stat' and 'chrom'.
        
    Returns:
        pd.DataFrame: A DataFrame of simulated variants sampled to match human-derived counts.
    """
    human_derived = data[data['driver_stat'] == 0]
    simulated_indels = data[data['driver_stat'] == 1]

    count_chrom = human_derived.groupby('chrom').size()
    sampled_simulated = []

    for chrom, count in count_chrom.items():
        sim_chrom = simulated_indels[simulated_indels['chrom'] == chrom]
        
        if count <= len(sim_chrom):
            sampled_simulated.append(sim_chrom.sample(n=count))#, random_state=42))
        else:
            sampled_simulated.append(sim_chrom)

    result = pd.concat(sampled_simulated, ignore_index=True)
    result = pd.concat([human_derived, result], ignore_index=True)
    return result


def train_and_evaluate_model(classifier, X_train_val, y_train_val, groups_train_val, X_test, y_test):
    """
    Train the model using Leave-One-Group-Out cross-validation and evaluate its performance.

    Parameters:
        classifier: The classifier model object.
        X_train_val (pd.DataFrame): Features of the training/validation set.
        y_train_val (pd.Series): Target variable of the training/validation set.
        groups_train_val (pd.Series): Groups corresponding to the training/validation set.
        X_test (pd.DataFrame): Features of the test set.
        y_test (pd.Series): Target variable of the test set.

    Returns:
        tuple: A tuple containing metrics, actual_targets, predicted_targets, final_model, test_accuracy, y_test_pred.
    """

    metrics = {
        "accuracy": [],
        "precision": [],
        "recall": [],
        "f1": [],
        "roc_auc": [],
    }
    # Leave-One-Group-Out cross-validation
    logo = LeaveOneGroupOut()
    actual_targets = np.array([])
    predicted_targets = np.array([])
    for train_index, val_index in logo.split(X_train_val, y_train_val, groups_train_val):
        
        model = classifier

        X_train, X_val = X_train_val.iloc[train_index], X_train_val.iloc[val_index]
        y_train, y_val = y_train_val.iloc[train_index], y_train_val.iloc[val_index]

        print(y_train.value_counts())

        if len(np.unique(y_val)) < 2:
            print(f"Skipping LOGO fold with chromosome {groups_train_val[val_index][0]} (only one class present).")
            continue  # Skip this fold
        # Random undersampling to balance the training data
        sampler = RandomUnderSampler(random_state=42)
        X_train_resampled, y_train_resampled = sampler.fit_resample(X_train, y_train)
        # print(X_train_resampled.shape)
        # Model fitting
        model.fit(X_train_resampled, y_train_resampled)
        print(X_train_resampled.shape)

        # Predict on validation set
        y_val_pred = model.predict(X_val)
        actual_targets = np.append(actual_targets, y_val)
        predicted_targets = np.append(predicted_targets, y_val_pred)

        # Evaluate and store metrics
        if len(np.unique(y_val))>1:
            y_val_pred_proba = model.predict_proba(X_val)[:, 1]
            metrics["roc_auc"].append(roc_auc_score(y_val, y_val_pred_proba))
        else:
            metrics["roc_auc"].append(np.nan)

        
        
        metrics["accuracy"].append(accuracy_score(y_val, y_val_pred))
        metrics["precision"].append(precision_score(y_val, y_val_pred))
        metrics["recall"].append(recall_score(y_val, y_val_pred))
        metrics["f1"].append(f1_score(y_val, y_val_pred))
        

    # Fit the final model on the entire training/validation set
    final_model = classifier
    final_sampler = RandomUnderSampler(random_state=42)
    X_train_val_resampled, y_train_val_resampled = final_sampler.fit_resample(X_train_val, y_train_val)
    final_model.fit(X_train_val_resampled, y_train_val_resampled)

    # Predict and evaluate on the test set
    y_test_pred = final_model.predict(X_test[X_train_val.columns.tolist()])
    test_accuracy = balanced_accuracy_score(y_test, y_test_pred)

    test_results = pd.DataFrame(
        {
            "balanced_accuracy": [test_accuracy],
            "precision": [precision_score(y_test, y_test_pred)],
            "recall": [recall_score(y_test, y_test_pred)],
            "f1": [f1_score(y_test, y_test_pred)],
            "roc_auc": [roc_auc_score(y_test, y_test_pred)],
        }
    )

    actual_predicted_targets_cross_val = (actual_targets, predicted_targets)
    actual_predicted_targets_test = (y_test, y_test_pred)
    actual_predicted = (actual_predicted_targets_cross_val, actual_predicted_targets_test)

    return metrics, final_model, test_results, actual_predicted

In [ ]:



class Data:
    def __init__(self, data_file: str):
        self.data_file = data_file
        self.data = pd.read_csv(data_file, sep=',')
        self.clean_data = drop_bias(self.data)
        self.sampled_data = select_equal_amounts(self.clean_data)
    def __str__(self):
         return str(self.sampled_data[self.sampled_data['driver_stat'] == 1].shape,self.sampled_data[self.sampled_data['driver_stat'] == 0].shape)
    


class Model(Data):
    def __init__(self,classifier: XGBClassifier, data_file: str):
        self.data = Data(data_file)

        self.classifier = classifier
        
    def target_and_label(self, data: pd.DataFrame):
        """Split data into train, val, test"""
        X = data.drop(columns=['chrom','pos', 'ref_allele', 'alt_allele', 'driver_stat'])

        y = data["driver_stat"]

        return X, y
    
    def get_idx(self, data, X, y):
        """Define chromosomal groups"""
        groups = data['chrom'].values
        unique_groups = np.unique(groups)
        np.random.shuffle(unique_groups)

        test_groups = []
        test_size = 0
        target_size = int(0.2 * len(data))

        for chrom in unique_groups:
            test_groups.append(chrom)
            test_size += (groups == chrom).sum()
            if test_size >= target_size:
                break
        print(test_groups)
        
        test_indices = np.isin(groups, test_groups)
        train_val_indices = ~test_indices
        test_indices, train_val_indices, groups

        X_train_val, y_train_val = X[train_val_indices], y[train_val_indices]
        X_test, y_test = X[test_indices], y[test_indices]
        groups_train_val = groups[train_val_indices]


        return X_train_val, y_train_val, X_test, y_test, groups_train_val

    def train_evaluate(self, data: pd.DataFrame):
        X, y = self.target_and_label(data)
        X_train_val, y_train_val, X_test, y_test, groups_train_val = self.get_idx(data, X, y)
        metrics, final_model, test_results, actual_predicted = train_and_evaluate_model(self.classifier, X_train_val, y_train_val, groups_train_val, X_test, y_test)

        return metrics, final_model, test_results, actual_predicted
    

data_file = '/Users/edatkinson/Repos/Modelling/merged_and_unmerged_data.csv'

candrivr = Model(XGBClassifier(), data_file)

metrics, final_model, test_results, actual_predicted = candrivr.train_evaluate(candrivr.data.sampled_data)


['chr6', 'chr5', 'chr14', 'chr22', 'chr10']
['chr1' 'chr1' 'chr1' ... 'chr9' 'chr9' 'chr9']
driver_stat
0    1689
1    1649
Name: count, dtype: int64
(3298, 467)
driver_stat
0    1689
1    1649
Name: count, dtype: int64
(3298, 467)
driver_stat
0    1749
1    1709
Name: count, dtype: int64
(3418, 467)
driver_stat
0    1839
1    1799
Name: count, dtype: int64
(3598, 467)
driver_stat
0    1809
1    1769
Name: count, dtype: int64
(3538, 467)
driver_stat
0    1753
1    1713
Name: count, dtype: int64
(3426, 467)
driver_stat
0    1741
1    1701
Name: count, dtype: int64
(3402, 467)
driver_stat
0    1828
1    1788
Name: count, dtype: int64
(3576, 467)
driver_stat
0    1621
1    1581
Name: count, dtype: int64
(3162, 467)
driver_stat
0    1727
1    1687
Name: count, dtype: int64
(3374, 467)
driver_stat
0    1824
1    1784
Name: count, dtype: int64
(3568, 467)
driver_stat
0    1833
1    1833
Name: count, dtype: int64
Skipping LOGO fold with chromosome chr21 (only one class present).
driver_stat
0

In [21]:
# X_train_val, y_train_val, X_test, y_test, groups_train_val = candrivr.get_idx(candrivr.data.sampled_data,*candrivr.target_and_label(candrivr.data.sampled_data))
print(test_results)

print(pd.DataFrame(metrics))
print(np.mean(metrics['accuracy']))

   balanced_accuracy  precision    recall        f1   roc_auc
0           0.982107   0.980198  0.984095  0.982143  0.982107
    accuracy  precision    recall        f1   roc_auc
0   0.991848   0.989189  0.994565  0.991870  0.999705
1   0.970109   0.977901  0.961957  0.969863  0.996219
2   0.991935   0.984127  1.000000  0.992000  0.999545
3   0.985294   0.971429  1.000000  0.985507  0.999135
4   0.984375   0.969697  1.000000  0.984615  0.996826
5   0.979167   0.967480  0.991667  0.979424  0.994444
6   0.984848   0.984848  0.984848  0.984848  0.998795
7   0.977778   0.977778  0.977778  0.977778  0.997531
8   0.978175   0.961686  0.996032  0.978558  0.998488
9   0.993151   0.986486  1.000000  0.993197  0.999531
10  0.979592   0.960784  1.000000  0.980000  0.997918
11  0.980159   0.976378  0.984127  0.980237  0.997480
12  0.993827   0.987805  1.000000  0.993865  1.000000
13  0.979675   0.991667  0.967480  0.979424  0.995704
14  0.948052   0.915663  0.987013  0.950000  0.994603
15  0.989130

In [ ]:
# print(classification_report(actual_predicted[1][0], actual_predicted[1][1]))


              precision    recall  f1-score   support

           0       0.99      0.97      0.98       479
           1       0.97      0.99      0.98       479

    accuracy                           0.98       958
   macro avg       0.98      0.98      0.98       958
weighted avg       0.98      0.98      0.98       958

